# Lab 19 · Optimisation 4: many people, one device

**Time:** 75 min (45 before tea, 30 after) · **Work in teams of 3**

**Where we are in the day:** Lab 18 gave us a small model file (986 MB for Q4_K_M) that runs on a CPU for one person; now many people talk to the device at once.
**Today's question:** how much memory does each user need, and how many people can one machine answer before they wait too long?
**What you will do:**
- Work out the memory one user needs (the KV cache) and see what batching does to waiting time and answers per second.
- Start a real serving engine (vLLM) with your fine-tuned model.
- Send it 1, 8, 32, 64, 128 and 256 people at once, and read the result.

**What you will see:** in our test run the answers per second rose from 11.7 (one person) to 559.9 (256 people), while the slow wait (p95) grew from 111 ms to 641 ms.
**Why it matters on a device:** the brief says how many people at once and how fast; the load test tells you whether the device can keep that promise.

**How to run this lab**
- Click **Kernel, Restart Kernel and Run All Cells**. Write your guess on paper before each cell marked **Predict**.
- The cells take about **1 to 2 minutes** in all (starting the server is most of it: 33 to 93 seconds in our test run).
- If a cell takes longer than that, wait until the time is over, then interrupt the kernel and ask the trainer.
- Before you start: Labs 17 and 18 must have been run (the first cell checks for the merged model they create), and the **setup script** (`setup_day3.sh`) must have finished, with SETUP DONE at the end of `/workspace/day3_setup.log`. The scoreboard should already have three rows.

**Words used in this lab:**
- **KV cache:** keeping your notes of the earlier conversation instead of re-reading everything for each new word. Every user has their own notes, and they take memory.
- **Latency:** how long one person waits for an answer. **Throughput:** how many answers per second the machine gives to everybody.
- **Batching:** sending several requests to the GPU together, as one group.
- **p50 and p95:** the typical wait (half of the requests were faster) and the slow wait (only 5 in 100 requests waited longer).

**The two numbers are not the same thing.**
- **Latency:** how long *one* request takes. A user feels this.
- **Throughput:** how many requests the machine finishes *per second*. The owner pays for this.

Serving many users at once usually raises throughput **and** raises each person's waiting time. The art is to find the point where both are acceptable.

**Python in this lab:** loops and dictionaries. The server and the load test are in `edge_serve.py`.

## Part 1: memory and batching
How much memory each user needs, and what batching does (19.1 to 19.3). It runs on the model that you made, on the GPU, with no server yet.

In [ ]:
import sys, os
for folder in ["..", "../.."]:                  # find the course folder (the one that holds slm_lab.py)
    if os.path.exists(os.path.join(folder, "slm_lab.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import torch
import pandas as pd
import edge_lab as el
import edge_nlu as nlu
from slm_lab import check, OUT
from transformers import AutoConfig
MERGED_FOLDER = str(OUT / "edge_merged")
golden = nlu.load_golden()
print("fine-tuned model:", MERGED_FOLDER, os.path.exists(MERGED_FOLDER))
print("scoreboard so far:", list(el.load_scoreboard().keys()))

**You should see** `True`. If it says `False`, run Labs 17 and 18 first: they create the merged model.

*In one sentence: this lab uses the model of Lab 18's merge, not the GGUF files.*

## 19.1 The KV cache: memory for every user
When a model writes its answer one token at a time, it needs some numbers about every earlier token. Recomputing them for each new token would be very slow, so the model **keeps them in memory**. This is the **KV cache**: it is like keeping your notes of the earlier conversation instead of re-reading everything for each reply. It grows with every token and **every user has their own**.

*In one sentence: the KV cache is the memory each user's conversation takes.*

> **Optional: if you are curious.** The numbers kept are the *keys and values* (K and V) of every earlier token, in every layer. That is why the cell below multiplies by 2 and by the number of layers.

We can compute its size from the model's settings.

In [ ]:
config = AutoConfig.from_pretrained(el.MODEL_ID)
layers = config.num_hidden_layers
kv_heads = config.num_key_value_heads
head_size = config.hidden_size // config.num_attention_heads
bytes_per_number = 2                             # 16-bit numbers
bytes_per_token = 2 * layers * kv_heads * head_size * bytes_per_number      # 2 = one K and one V
print("layers:", layers, " key-value heads:", kv_heads, " head size:", head_size)
print("KV cache per token:", bytes_per_token, "bytes =", round(bytes_per_token / 1000, 1), "KB")

**You should see** about 28 KB per token (28,672 bytes in our test run).

**Predict.** Write on paper how many MB one user needs if their request is 50 tokens long, and how many MB if it is 450 tokens long (the best prompt of Lab 16). Use 28 KB per token. The next cell adds 25 tokens for the answer, so its numbers will be a little higher than yours.

In [ ]:
for name, tokens in [("one-line prompt + answer", 50 + 25), ("best prompt of Lab 16 + answer", 464 + 25)]:
    one_user_mb = tokens * bytes_per_token / 1e6
    print(name, "->", tokens, "tokens ->", round(one_user_mb, 1), "MB per user,", round(64 * one_user_mb), "MB for 64 users")

**You should see** that a long prompt costs about six times as much memory per user as the one-line prompt (in our test run: 2.2 MB against 14.0 MB). This is another reason why the fine-tuned model with a short prompt is cheaper to serve: **every token of the prompt is paid by every user, every time.**

**How many users fit?** Our memory budget was about 7.6 GB (Lab 17). The weights take part of it, and the cache takes the rest.

In [ ]:
BUDGET_GB = 7.6
WEIGHTS_GB = 3.1                                 # the model in 16 bits
free_gb = BUDGET_GB - WEIGHTS_GB
for name, tokens in [("one-line prompt", 75), ("long prompt", 489)]:
    users = free_gb * 1e9 / (tokens * bytes_per_token)
    print(name, "->", round(users), "users fit in the cache memory")

**You should see** that **memory is not the limit** for a model of this size and a short prompt: in our test run about 2,093 users fit with the one-line prompt and about 321 with the long prompt. The limit is **speed**: how many requests the GPU can finish each second. That is what the load test measures.

In [ ]:
check(bytes_per_token == 28672, "Checkpoint: the KV cache is 28,672 bytes per token, as in our test run.",
      "Your number is different. That is fine if the model is not Qwen 1.5B; otherwise run the cells of 19.1 again.")

## 19.2 A context budget
Every model has a **maximum context**: the prompt and the answer together must fit. A bigger context needs a bigger cache. Write the function that tells you how many more tokens of text you can still add to a prompt, keeping space for the answer.

In [ ]:
def room_left(limit, used, reserve):
    return 0                                     # TODO: tokens still free = limit - used - reserve (never below 0)

print(room_left(2048, 1800, 100))
print(room_left(2048, 2000, 100))

In [ ]:
check(room_left(2048, 1800, 100) == 148 and room_left(2048, 2000, 100) == 0,
      "Right: the answer needs space too. 2048 - 1800 - 100 is 148, and when the prompt is already too long the room is 0, not a negative number.",
      "Free tokens = limit - used - reserve. If that is below zero, the answer is 0.")

## 19.3 Batching: many requests in one go
A GPU is built to do many things at once. Doing **one request** at a time wastes it. **Batching** sends several requests together. Everybody in the batch **waits for the whole batch**, but the machine finishes more requests per second.

**Predict.** Write on paper: if we send 1, 8 and 32 messages together, what happens to (a) the seconds each person waits, and (b) the answers per second? Write "up a lot", "up a little" or "down" for each.

In [ ]:
tok, model = el.load_edge_model(MERGED_FOLDER)
prompt_texts = []
for case in golden:
    prompt_texts.append(el.make_prompt_text(tok, nlu.SHORT_SYSTEM, case["text"]))
el.ask_many(tok, model, prompt_texts[:2])        # one practice batch, not counted: the first batch on a GPU is always slow
batch_rows = []
for batch_size in [1, 8, 32, 64]:
    batch = []
    for number in range(batch_size):
        batch.append(prompt_texts[number % len(prompt_texts)])
    result = el.ask_many(tok, model, batch, batch_size=batch_size)
    batch_rows.append({"batch size": batch_size, "seconds each person waits": round(result["seconds"], 2),
                       "answers per second": round(batch_size / result["seconds"], 1)})
pd.DataFrame(batch_rows)

**You should see** the waiting time stay **small** while the answers per second rise **a lot**. In our test run: a wait of 0.23 s for 1 message and 0.36 s for 32, while the answers per second rose from 4.3 to 88.9 (and 172.5 at 64). A batch of 32 is nowhere near 32 times slower than a batch of 1. This is why batching is the first tool of every serving system.

*If it looks different:* the timing depends on your GPU. The pattern (wait grows slowly, answers per second grow fast) should be the same.

The weakness of this **static batching**: everybody waits for the **slowest** answer in the batch, and a new request cannot join until the batch is finished. A real serving engine fixes this with **continuous batching**.

In [ ]:
del model
torch.cuda.empty_cache()

In [ ]:
check(len(batch_rows) == 4, "Checkpoint: you measured batches of 1, 8, 32 and 64.",
      "Run the batching cell of 19.3 again. If it says out of memory, restart the kernel and run all cells.")

# Part 2: the real server and the load test
**You now have:** the memory one user needs, and proof that batching raises answers per second. Now we run a real serving engine and send it many people at once.

## 19.4 A real serving engine
**vLLM** is a serving engine used in production. Two ideas make it efficient:
- **Continuous batching:** as soon as one request finishes, a waiting one takes its place. Nobody waits for the slowest.
- **Paged KV cache and prefix caching:** the cache is kept in small blocks, so no memory is wasted, and the **fixed beginning of the prompt** (our system message) is computed **once** and reused by every request.

vLLM needs its own copy of PyTorch, so it lives in a **separate environment**. The setup script `setup_day3.sh` (started at the beginning of the day) installs it. The next cell only checks.

In [ ]:
import edge_serve as es
print("vLLM installed:", os.path.exists(es.VENV_PYTHON))

**You should see** `True`. If it says `False`, the install has not finished: run `bash setup_day3.sh` in a terminal if you have not, and wait for SETUP DONE in `/workspace/day3_setup.log`.

We start the server with our fine-tuned model. We let it use **30 per cent of the GPU memory**, to respect our budget.

In [ ]:
server = es.start_server(MERGED_FOLDER, memory_share=0.3)
seconds = es.wait_until_ready(server)
print("the server is ready after", seconds, "seconds")
reply, wait = es.ask_server(nlu.SHORT_SYSTEM, "turn on the kitchen light")
print(reply, "|", round(1000 * wait), "ms")

In [ ]:
check(len(reply) > 0, "Checkpoint: the server is running and answered one request.",
      "The server gave no answer. Run the start cell again and wait for the ready line; if it fails twice, tell the trainer.")

**You should see** the server ready after a minute or two (it loads the model once), and then an answer to one request in a fraction of a second. In our test run the first start took 93 seconds and the second 33 seconds.

## 19.5 The load test
Now we send **1, 8, 32, 64, 128 and 256 people at the same time**. Write down the p95 and the answers per second for 1, 32 and 256 people. Each person sends several messages, one after the other. We measure how long each answer took: the **typical wait** (p50: half of the requests were faster) and the **slow wait** (p95: only 5 in 100 requests waited longer). We also check the **accuracy**, because a fast server that changes the answers is not an improvement.

In [ ]:
load_rows = []
for clients in [1, 8, 32, 64, 128, 256]:
    result = es.load_test(clients, nlu.SHORT_SYSTEM, golden)
    load_rows.append({"people at once": clients, "typical wait ms (p50)": result["p50_ms"], "slow wait ms (p95)": result["p95_ms"],
                      "answers per second": result["per_second"], "exact %": result["exact_percent"]})
load_table = pd.DataFrame(load_rows)            # pandas only shows the finished table
load_table

In [ ]:
check(len(load_rows) == 6, "Checkpoint: the load test has 6 rows (1 to 256 people).",
      "Run the load test cell again. If the server stopped, run the start cell of 19.4 first.")

**You should see** the **answers per second rise with the number of people**, and the **waiting time rise** as well. At some number of people the answers per second stop rising and only the waiting time grows: the machine is full. In our test run on an RTX 4090: one person got a slow wait of 111 ms and 11.7 answers per second; 128 people got 383 ms and 484.5; 256 people got 641 ms and 559.9. The waiting time rose only a little up to 64 people, so the full point is further on: look at 128 and 256. Where is the point in your table?

*If it looks different:* the numbers depend on your GPU. The accuracy column should stay close to 89 per cent in every row (a fast server must not change the answers).

In [ ]:
import matplotlib.pyplot as plt
figure, axis = plt.subplots(figsize=(6, 3.2))
axis.plot(load_table["people at once"], load_table["slow wait ms (p95)"], marker="o", label="slow wait ms (p95)")
axis.set_xlabel("people at once")
axis.set_ylabel("milliseconds")
second_axis = axis.twinx()
second_axis.plot(load_table["people at once"], load_table["answers per second"], marker="s", color="green", label="answers per second")
second_axis.set_ylabel("answers per second")
plt.title("Waiting time and throughput")
plt.show()

## 19.6 Do we meet the brief?
Lab 16 set a target for the number of people at once and for the answer time. Compare.

In [ ]:
BRIEF = {"answer_ms": 500, "users_at_once": 32}    # 🔧 the targets from Lab 16
row = load_rows[0]
for candidate in load_rows:
    if candidate["people at once"] == BRIEF["users_at_once"]:
        row = candidate
print("with", BRIEF["users_at_once"], "people at once: slow wait", row["slow wait ms (p95)"], "ms, target", BRIEF["answer_ms"], "ms")
if row["slow wait ms (p95)"] <= BRIEF["answer_ms"]:
    print("the speed target is met")
else:
    print("the speed target is NOT met")

**How many people can one device serve?** If each person speaks once every 30 seconds, then a server that answers N requests per second serves about **N × 30 people**.

In [ ]:
SECONDS_BETWEEN_REQUESTS = 30                   # 🔧 how often one person speaks
best_row = load_rows[0]
for candidate in load_rows:
    if candidate["slow wait ms (p95)"] <= BRIEF["answer_ms"]:
        best_row = candidate                    # the busiest row that still meets the waiting target
print("busiest row that meets the target:", int(best_row["people at once"]), "people at once")
print("answers per second:", best_row["answers per second"])
print("people one server can serve:", round(best_row["answers per second"] * SECONDS_BETWEEN_REQUESTS))

**You should see** a number of people per server. This is the **"how many users per device"** answer. In our test run the busiest row that met 500 ms was 128 people at once, with 484.5 answers per second, so about 14,535 people per server at one request every 30 seconds. This is a rough estimate, not a promise. Change `SECONDS_BETWEEN_REQUESTS` (a voice assistant used all evening is spoken to more often) and see how it changes.

In [ ]:
es.stop_server(server)
el.save_row("4 served", {"what": "vLLM, one-line prompt", "people_at_once": int(best_row["people at once"]),
                        "p95_ms": int(best_row["slow wait ms (p95)"]), "answers_per_second": float(best_row["answers per second"]),
                        "exact_percent": float(best_row["exact %"])})
print(el.load_scoreboard()["4 served"])
check("4 served" in el.load_scoreboard(), "Checkpoint: the fourth row of the scoreboard is saved.",
      "The row is missing. Run the cell above again (the server must have been started first).")

## What you take away
- The KV cache is the memory each user's conversation takes, so a short prompt is cheaper for everybody.
- Latency is one person's wait and throughput is the machine's answers per second. Batching raises throughput a lot for a small rise in waiting time; continuous batching is better than static batching.
- Always read the load test together with accuracy. "Users per device" is answers per second times the seconds between requests.

## Check your understanding
1. Why does the KV cache grow with the number of users?
*✍️ Your answer:* 
2. In the load test, throughput stopped rising while the waiting time kept rising. What does it mean?
*✍️ Your answer:* 
3. Does batching always lower the waiting time of one request?
*✍️ Your answer:* 
4. How does a shorter prompt help serve more users?
*✍️ Your answer:* 

## Next
**Lab 20:** the model is small and fast, and still about 1 message in 10 is wrong. What should the assistant do then?